# 프로토콜 v1.1 — E2c: CruiseBench 공개 코드를 직접 실행 (탐색 분석, GPU)

CruiseBench(커밋 `ef26d40`)의 코드를 **원 설정 그대로** 9개 서브셋에서 실행해, 시험 창마다 예측을 저장합니다. 채점과 PI-BDP와의 짝지은 검정은 다음 노트북(E2c 채점, CPU)에서 합니다.

- **A단계 — 재현 확인 (v1.1 3.4절):** DS01·DS04에서 LSTM·TSMixer를 시드 1–5로 학습하고, 원 논문처럼 **매 epoch 시험 평가 후 최고 epoch 값**을 계산해 논문 표 6.1과 비교합니다. 판정: 우리 평균이 논문 평균 ± max(2 × 논문 SD, 0.3) 안이면 "재현됨".
- **B단계 — 본 실행 (v1.1 3.2절):** 9개 서브셋 × LSTM·GRU·TCN·TSMixer × 시드 1–5. 학습 중에는 시험 데이터를 보지 않고, **50번째 epoch이 끝난 뒤 한 번만** 시험 창을 예측합니다.
- **계산량 규칙:** A단계 첫 실행(DS01, LSTM, 시드 1)의 첫 epoch 학습 시간으로 B단계 180회 전체 시간을 추정해, 40 GPU시간을 넘으면 B단계 시드를 3회(1–3)로 줄입니다. 성능 지표는 보지 않고 시간만 씁니다.
- **원 코드는 고치지 않습니다.** 원 코드의 모델·학습 함수를 그대로 불러 쓰고, 바뀐 것은 시드 지정, 시험 평가 시점, 창별 예측 저장(엔진·사이클 번호 기록)뿐입니다.

**실행 전:** 이 노트북을 커밋한 주소를 `RUN_COMMIT_URL`에 넣으세요. **런타임:** GPU(A100 또는 L4 권장), 고용량 RAM. 끊겨도 다시 실행하면 끝난 실행은 건너뜁니다.

In [ ]:
# === 1. 설정 ===
import os, sys, json, glob, time, hashlib, subprocess, zipfile, re, shutil
import numpy as np, pandas as pd
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/PI-BDP_NCMAPSS'; WORK = '/content/ncmapss_raw'
else:
    ROOT = os.environ.get('NCM_ROOT', './ncm'); WORK = os.environ.get('NCM_WORK', './ncmapss_raw')
OUT = f'{ROOT}/addendum_v11/e2c_cruise'; os.makedirs(f'{OUT}/preds', exist_ok=True); os.makedirs(f'{OUT}/logs', exist_ok=True)
os.makedirs(WORK, exist_ok=True)
ADDENDUM_COMMIT = 'fb8db254edb1fa613a00c4c22b031c7c7515489d'      # 프로토콜 v1.1 커밋
RUN_COMMIT_URL = ''                                                # 이 노트북을 커밋한 주소
assert re.match(r'https://github\.com/.+/commit/[0-9a-f]{7,40}$', RUN_COMMIT_URL), '이 노트북을 먼저 커밋하고 주소를 넣으세요.'
SUBSETS = ['DS01-005', 'DS02-006', 'DS03-012', 'DS04', 'DS05', 'DS06', 'DS07', 'DS08a-009', 'DS08c-008']   # CruiseBench 순서
CAPS = dict(zip(SUBSETS, [38, 34, 34, 43, 38, 41, 39, 34, 28]))
MODELS = ['LSTM', 'GRU', 'TCN', 'TSMixer']
import torch
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('장치:', DEV, torch.cuda.get_device_name(0) if DEV == 'cuda' else '')

In [ ]:
# === 2. N-CMAPSS 내려받기 (Phase 2b와 같은 파일인지 해시로 확인) ===
URL = 'https://phm-datasets.s3.amazonaws.com/NASA/17.+Turbofan+Engine+Degradation+Simulation+Data+Set+2.zip'
MANUAL_ZIP = ''   # 자동 내려받기가 안 되면 드라이브에 올린 zip 경로
def h5_list(): return sorted(glob.glob(os.path.join(WORK, '**', '*.h5'), recursive=True))
if not h5_list():
    zpath = os.path.join(WORK, 'ncmapss.zip')
    if MANUAL_ZIP: subprocess.run(['cp', MANUAL_ZIP, zpath])
    elif not (os.path.exists(zpath) and zipfile.is_zipfile(zpath)):
        subprocess.run(['wget', '-c', '-q', '--show-progress', '-O', zpath, URL])
    done = set()
    while True:
        zs = [z for z in glob.glob(os.path.join(WORK, '**', '*.zip'), recursive=True) if z not in done]
        if not zs: break
        for z in zs:
            done.add(z)
            if zipfile.is_zipfile(z):
                with zipfile.ZipFile(z) as f: f.extractall(os.path.splitext(z)[0] + '_x')
    for z in glob.glob(os.path.join(WORK, '**', '*.zip'), recursive=True): os.remove(z)
H5 = {os.path.basename(p).replace('.h5', '').replace('N-CMAPSS_', ''): p for p in h5_list()}
REFH = json.load(open(f'{ROOT}/phase2b/sha256_h5.json'))
FLAT = os.path.join(WORK, 'flat'); os.makedirs(FLAT, exist_ok=True)
for s in SUBSETS:
    assert os.path.getsize(H5[s]) == REFH[s]['bytes'], f'{s}: 파일 크기가 Phase 2b와 다름'
    h = hashlib.sha256()
    with open(H5[s], 'rb') as f:
        for b in iter(lambda: f.read(1 << 24), b''): h.update(b)
    assert h.hexdigest() == REFH[s]['sha256'], f'{s}: 해시가 Phase 2b와 다름'
    dst = os.path.join(FLAT, f'N-CMAPSS_{s}.h5')
    if not os.path.exists(dst): os.symlink(H5[s], dst)
print('h5 9개 해시 확인 완료')

In [ ]:
# === 3. CruiseBench 고정 커밋 받기, 원 코드 불러오기 ===
CB = '/content/CruiseBench' if IN_COLAB else os.environ.get('CB_DIR', './CruiseBench')
CB_COMMIT = 'ef26d40'
if not os.path.exists(CB):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/NostalgiaJohn/CruiseBench', CB], check=True)
subprocess.run(['git', '-C', CB, 'checkout', '-q', CB_COMMIT], check=True)
CB_HEAD = subprocess.run(['git', '-C', CB, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert CB_HEAD.startswith(CB_COMMIT)
assert not subprocess.run(['git', '-C', CB, 'status', '--porcelain'], capture_output=True, text=True).stdout.strip(), '원 코드가 바뀌어 있음'
sys.path.insert(0, CB)
import main as cbmain, data_load as cbload
MASK = f'{CB}/dataset/cruise_mask/common_altitude_cruise_mask.h5'
print('CruiseBench', CB_HEAD)

class IdLoader(cbload.Data_Loader):
    # 원 코드의 창 생성은 그대로 쓰고, 각 창의 (엔진, 사이클, 창 번호)만 따로 기록한다.
    def _make_windows(self, data_df, feature_cols, window_size, stride, split):
        ids = []
        for (u, cy), g in data_df.groupby(['unit', 'cycle'], sort=False):
            if len(g) < window_size: continue
            ids += [(int(u), int(cy), k) for k in range(len(range(0, len(g) - window_size + 1, stride)))]
        X, y = super()._make_windows(data_df, feature_cols, window_size, stride, split)
        assert len(ids) == len(y), 'id 수와 창 수가 다름'
        self.ids = getattr(self, 'ids', {}); self.ids[split] = np.array(ids, dtype=np.int32)
        return X, y

def cb_args(subset, model, seed):
    # 원 코드의 명령행 기본값을 그대로 쓰고, README의 eta5-W256-S10 설정만 지정한다.
    argv = ['--data_folder', FLAT, '--dataset_choice', str(SUBSETS.index(subset)), '--cruise_mask_path', MASK,
            '--window_size', '256', '--stride', '10', '--max_rul', str(CAPS[subset]), '--model_name', model,
            '--all_channels', '18', '--pred_len', '1', '--num_epochs', '50', '--train_batch_size', '512',
            '--test_batch_size', '512', '--learning_rate', '3e-4', '--seed', str(seed), '--optimizer_name', 'Adam',
            '--scheduler_name', 'None', '--device', DEV, '--run_name', f'{subset}_{model}_s{seed}', '--logging_dir', f'{OUT}/logs']
    old = sys.argv; sys.argv = ['main.py'] + argv
    try: return cbmain.parse_args()
    finally: sys.argv = old

In [ ]:
# === 4. 한 번의 학습 (원 코드 main()·Processor와 같은 순서) ===
DATA = {}
def get_data(subset):
    if subset not in DATA:
        DATA.clear()   # 메모리 절약: 한 번에 한 서브셋만
        t = time.time()
        DATA[subset] = IdLoader(data_path=f'{FLAT}/N-CMAPSS_{subset}.h5', window_size=256, stride=10,
                                max_rul=CAPS[subset], cruise_mask_path=MASK, num_workers=2)
        d = DATA[subset]; print(f'{subset}: 학습 창 {len(d.train_label):,}, 시험 창 {len(d.test_label):,} ({time.time() - t:.0f}초)')
    return DATA[subset]

def predict(model, loader):
    model.eval(); out = []
    with torch.no_grad():
        for xb, _ in loader: out.append(model(xb.to(DEV)).float().cpu().numpy().reshape(-1))
    return np.concatenate(out)

def run_one(subset, model_name, seed, per_epoch_eval=False, time_first_epoch=False):
    a = cb_args(subset, model_name, seed)
    cbmain.set_seed(a.seed)                                   # main(): 시드 → 모델 → 최적화기 → 스케줄러 → 데이터
    a.device = torch.device(a.device); a.model = cbmain.build_model(a)
    a.optimizer = cbmain.build_optimizer(a); a.scheduler = cbmain.build_scheduler(a)
    d = get_data(subset); pin = DEV == 'cuda'
    tr = d.getTrainDataloader(batch_size=a.train_batch_size, shuffled=True, memory_pinned=pin)
    te = d.getTestDataloader(batch_size=a.test_batch_size, shuffled=False, memory_pinned=pin)
    model = a.model.to(a.device); crit = torch.nn.MSELoss(); hist = []; t_first = None; t0 = time.time()
    for ep in range(a.num_epochs):                           # Processor.train()과 같은 학습 고리
        model.train(); tot = 0.0; te0 = time.time()
        for xb, yb in tr:
            xb = xb.to(a.device, non_blocking=True); yb = yb.to(a.device, non_blocking=True)
            loss = crit(model(xb), yb); tot += loss.detach().float().item()
            loss.backward(); a.optimizer.step(); a.optimizer.zero_grad(set_to_none=True)
        a.scheduler.step()
        if ep == 0: t_first = time.time() - te0
        if time_first_epoch and ep == 0: return dict(t_first_epoch=t_first, n_train=len(d.train_label))
        rec = dict(epoch=ep + 1, train_loss=tot / len(tr))
        if per_epoch_eval:                                    # A단계(재현 확인)에서만, 원 논문처럼 매 epoch 평가
            p = predict(model, te); rec['test_rmse'] = float(np.sqrt(np.mean((p - d.test_label) ** 2)))
        hist.append(rec)
    p = predict(model, te)                                    # 50번째 epoch 뒤 한 번
    ids = d.ids['test']
    df = pd.DataFrame({'subset': subset, 'unit': ids[:, 0], 'cycle': ids[:, 1], 'win': ids[:, 2],
                       'label_capped': d.test_label, 'pred': p, 'model': model_name, 'seed': seed})
    return dict(pred=df, hist=pd.DataFrame(hist), t_first_epoch=t_first, t_total=time.time() - t0, n_train=len(d.train_label))

In [ ]:
# === 5. A단계: 재현 확인 (DS01·DS04 × LSTM·TSMixer × 시드 1–5) ===
PAPER = {('DS01-005', 'LSTM'): (10.30, 1.47), ('DS01-005', 'TSMixer'): (2.25, 0.38),
         ('DS04', 'LSTM'): (13.97, 0.03), ('DS04', 'TSMixer'): (4.72, 0.16)}      # 논문 표 6.1 (평균, SD)
RA = f'{OUT}/replication'; os.makedirs(RA, exist_ok=True)
timing_f = f'{OUT}/timing.json'
for (s, m) in PAPER:
    for sd in range(1, 6):
        f = f'{RA}/{s}_{m}_s{sd}_hist.csv'
        if os.path.exists(f): continue
        r = run_one(s, m, sd, per_epoch_eval=True)
        r['hist'].to_csv(f, index=False)
        r['pred'].to_csv(f'{RA}/{s}_{m}_s{sd}_final_pred.csv.gz', index=False)
        if not os.path.exists(timing_f) and (s, m, sd) == ('DS01-005', 'LSTM', 1):
            json.dump(dict(t_first_epoch=r['t_first_epoch'], n_train_DS01=r['n_train'], device=torch.cuda.get_device_name(0) if DEV == 'cuda' else 'cpu'),
                      open(timing_f, 'w'), indent=1)
        print(s, m, sd, f"최고 epoch RMSE {r['hist'].test_rmse.min():.3f}, 마지막 {r['hist'].test_rmse.iloc[-1]:.3f}, {r['t_total'] / 60:.1f}분")
rows = []
for (s, m), (mu, sdv) in PAPER.items():
    H = [pd.read_csv(f'{RA}/{s}_{m}_s{k}_hist.csv') for k in range(1, 6)]
    best = np.array([h.test_rmse.min() for h in H]); last = np.array([h.test_rmse.iloc[-1] for h in H])
    tol = max(2 * sdv, 0.3)
    rows.append(dict(subset=s, model=m, paper_mean=mu, paper_sd=sdv, ours_best_mean=best.mean(), ours_best_sd=best.std(ddof=1),
                     ours_last_mean=last.mean(), tolerance=tol, reproduced=bool(abs(best.mean() - mu) <= tol)))
REP = pd.DataFrame(rows); REP.to_csv(f'{OUT}/e2c_replication.csv', index=False)
print(REP.round(3).to_string(index=False))
print('재현 판정:', '모두 재현됨' if REP.reproduced.all() else '재현되지 않은 항목 있음 → 원인 확인 후 한 번 더 (v1.1 3.4절)')

In [ ]:
# === 6. 계산량 규칙: B단계 시드 수 결정 (시간만 사용) ===
T = json.load(open(timing_f))
n_tr = {}
for s in SUBSETS:      # 학습 창 수는 마스크로 계산 (데이터를 다시 읽지 않음)
    import h5py
    with h5py.File(MASK, 'r') as h: R = h['cycles'][()]
    sel = (np.char.decode(R['dataset']) == s) & (np.char.decode(R['split']) == 'dev') & R['detected'].astype(bool)
    L = R['end_sample'][sel] - R['start_sample'][sel] + 1
    n_tr[s] = int(np.where(L >= 256, (L - 256) // 10 + 1, 0).sum())
est_h = T['t_first_epoch'] * 50 * len(MODELS) * 5 * sum(n_tr.values()) / T['n_train_DS01'] / 3600
N_SEEDS = 5 if est_h <= 40 else 3
dec = dict(t_first_epoch_s=T['t_first_epoch'], device=T['device'], est_gpu_hours_5seeds=est_h, n_seeds=N_SEEDS,
           rule='v1.1 3.2: >40 GPU-h -> 3 seeds for all models')
json.dump(dec, open(f'{OUT}/seed_decision.json', 'w'), indent=1); print(dec)

In [ ]:
# === 7. B단계: 본 실행 (마지막 epoch만 평가, 끊기면 다시 실행) ===
dec = json.load(open(f'{OUT}/seed_decision.json')); N_SEEDS = dec['n_seeds']
log = f'{OUT}/run_log.csv'
for s in SUBSETS:
    for m in MODELS:
        for sd in range(1, N_SEEDS + 1):
            f = f'{OUT}/preds/{s}_{m}_s{sd}.csv.gz'
            if os.path.exists(f): continue
            r = run_one(s, m, sd)
            r['pred'].to_csv(f, index=False); r['hist'].to_csv(f'{OUT}/logs/{s}_{m}_s{sd}_trainloss.csv', index=False)
            pd.DataFrame([dict(subset=s, model=m, seed=sd, minutes=r['t_total'] / 60, n_train=r['n_train'],
                               finished=time.strftime('%Y-%m-%d %H:%M:%S'))]).to_csv(log, mode='a', header=not os.path.exists(log), index=False)
            print(s, m, sd, f"{r['t_total'] / 60:.1f}분")
print('B단계 완료:', len(glob.glob(f'{OUT}/preds/*.csv.gz')), '/', len(SUBSETS) * len(MODELS) * N_SEEDS)

In [ ]:
# === 8. 기록과 해시 (채점은 다음 노트북) ===
files = sorted(glob.glob(f'{OUT}/**/*.*', recursive=True))
H = {os.path.relpath(p, OUT): hashlib.sha256(open(p, 'rb').read()).hexdigest() for p in files if not p.endswith('sha256_e2c_cruise.json')}
json.dump(dict(addendum=ADDENDUM_COMMIT, run_commit=RUN_COMMIT_URL, cruisebench=CB_HEAD, files=H),
          open(f'{OUT}/sha256_e2c_cruise.json', 'w'), indent=1)
print('파일', len(H), '개 해시 기록')